In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../data/Mall_Customers.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#بررسی مقادیر گم شده
df.isnull().sum()

In [ ]:
# توزیع متغیرهای عددی

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']):
    sns.histplot(df[col], kde=True, ax=ax, color='#0099cc')
    ax.set_title(f'توزیع {col}')
plt.tight_layout()
plt.savefig('../images/distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# توزیع جنسیت مشتریان

plt.figure(figsize=(5, 4))
sns.countplot(x='Gender', data=df, palette=['#0099cc', '#ff6f61'])
plt.title('توزیع جنسیت مشتریان')
plt.show()

print(df['Gender'].value_counts(normalize=True).round(2))

In [ ]:
# همبستگی بین ویژگی‌های عددی

plt.figure(figsize=(5, 4))
numeric_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('ماتریس همبستگی')
plt.show()

In [ ]:
%%writefile ../src/preprocessor.py

import pandas as pd
from sklearn.preprocessing import StandardScaler


class Preprocessor:
    """Cleans the raw mall-customer dataframe and prepares it for KMeans."""

    FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.scaler = StandardScaler()

    def handle_not_numeric(self):
        # هماهنگ با انکودینگ استفاده‌شده در بخش مدل‌سازی نوت‌بوک: مرد=1، زن=0
        self.df['Gender'] = self.df['Gender'].map({'Male': 1, 'Female': 0})

    def drop_unused_columns(self):
        if 'CustomerID' in self.df.columns:
            self.df = self.df.drop(columns=['CustomerID'])

    def scale_numeric(self):
        self.df[self.FEATURES] = self.scaler.fit_transform(self.df[self.FEATURES])

    def transform(self):
        self.handle_not_numeric()
        self.drop_unused_columns()
        self.scale_numeric()
        return self.df[['Gender'] + self.FEATURES]

In [ ]:
import sys
sys.path.append('../src')
from preprocessor import Preprocessor

df_scaled = Preprocessor(df).transform()
df_scaled.head()

In [ ]:
# پیدا کردن تعداد بهینه‌ی خوشه‌ها

features = Preprocessor.FEATURES

inertia = []
silhouette = []
K = range(2, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(df_scaled[features])
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(df_scaled[features], labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(K, inertia, marker='o', color='#0099cc')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('تعداد خوشه‌ها (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K, silhouette, marker='o', color='#ff6f61')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('تعداد خوشه‌ها (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('../images/elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()

best_k = list(K)[int(np.argmax(silhouette))]
print(f'بهترین تعداد خوشه بر اساس بیشترین Silhouette Score: k = {best_k}')

In [ ]:
model = KMeans(
    n_clusters=best_k,
    init='k-means++',
    n_init=12,
    max_iter=400,
    random_state=RANDOM_STATE,
)
model.fit(df_scaled[features])

df['Cluster'] = model.labels_
df_scaled['Cluster'] = model.labels_

In [ ]:
sil_score = silhouette_score(df_scaled[features], df_scaled['Cluster'])
final_score = 100 * (sil_score + 1) / 2
print(f'Silhouette Score: {sil_score:.4f}')
print(f'نمره‌ی نهایی (۰ تا ۱۰۰): {final_score:.2f}')

In [ ]:
# نمایش بصری خوشه‌ها

plt.figure(figsize=(7, 5.5))
palette = sns.color_palette('tab10', n_colors=best_k)
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Cluster',
    palette=palette,
    s=70,
)
plt.title('خوشه‌بندی مشتریان بر اساس درآمد سالانه و امتیاز خرج‌کرد')
plt.legend(title='خوشه', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../images/cluster_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', hue='Cluster', palette=palette, s=60, ax=axes[0], legend=False)
axes[0].set_title('سن در مقابل امتیاز خرج‌کرد')

sns.scatterplot(data=df, x='Age', y='Annual Income (k$)', hue='Cluster', palette=palette, s=60, ax=axes[1])
axes[1].set_title('سن در مقابل درآمد سالانه')
axes[1].legend(title='خوشه', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../images/cluster_age_views.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# تفسیر کسب‌وکاری خوشه‌ها

cluster_profile = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)
cluster_profile['Count'] = df.groupby('Cluster').size()
cluster_profile.sort_values('Annual Income (k$)', ascending=False)

In [ ]:
#ذخیره مدل 

joblib.dump(model, '../models/kmeans_model.joblib')
df.to_csv('../data/mall_customers_with_clusters.csv', index=False)
print('مدل و خروجی خوشه‌بندی ذخیره شد.')